# Build and Load the sample data
Run every cell from top to bottom. When it finishes, your warehouse holds the raw
data and you are ready to start running dbt.

In [ ]:
import sys
from pathlib import Path


def find_project_root(start):
    # Walk up the folder tree until we find pyproject.toml. This lets the notebook
    # work no matter which directory Jupyter happened to start in.
    for folder in [start, *start.parents]:
        if (folder / 'pyproject.toml').exists():
            return folder
    raise FileNotFoundError('Could not find the project root above this notebook.')


PROJECT_ROOT = find_project_root(Path.cwd())

# Make the helper modules in data_setup importable by name.
sys.path.insert(0, str(PROJECT_ROOT / 'data_setup'))

from generate_sample_data import generate_all
from load_raw_data import load_all

RAW_DATA_DIR = PROJECT_ROOT / 'raw_data'
print('Project root:', PROJECT_ROOT)

## Generate the raw CSV files
This writes four files into the `raw_data` folder. Because the generator uses a fixed random seed, you get the same rows every time.

In [ ]:
tables = generate_all(RAW_DATA_DIR)

## Load the data into Postgres
This reads the CSVs and writes them into the `raw` schema. Connection settings come from your `.env` file, so no password appears in this notebook.

In [ ]:
load_all(RAW_DATA_DIR)

## Confirm the load

In [ ]:
import pandas as pd
from load_raw_data import get_engine

engine = get_engine()
counts = pd.read_sql(
    """
    select 'raw_customers' as table_name, count(*) as row_count from raw.raw_customers
    union all select 'raw_products', count(*) from raw.raw_products
    union all select 'raw_orders', count(*) from raw.raw_orders
    union all select 'raw_order_items', count(*) from raw.raw_order_items
    """,
    engine,
)
engine.dispose()
counts